In [1]:
"""
End-user LPG price allocation per pixel (Nigeria, EPSG:3857)
Raster-to-raster implementation.

Updated 2026-04:
- Reads 8 bands (includes walk/car distances).
- Collection cost is per-trip (per 12.5 kg cylinder), not annualised.
- Uses actual path distances for vehicle operating cost.
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from rasterio.warp import reproject, Resampling
import rioxarray
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import transform

DATA_DIR = Path("dataset_big")

# Input from 4.2 (multilayer raster with 8 bands)
SOURCE_PIXEL_RASTER = DATA_DIR / "huff_preferred_distributor_per_pixel.tif"

# Cost source from 4.4 / 4.5
COST_SOURCE_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
RESELLER_ID_COL = "id_res&fil"
RESELLER_COST_COL = "cost_res_kg_out"
FILLING_LAYER = "filling"
FILLING_COST_COL = "cost_fil_kg_out"

# Input from 4.3 and base demography
LPG_USE_SHARE_RASTER = DATA_DIR / "lpg_use_share.tif"
POPULATION_RASTER = DATA_DIR / "Population.tif"

# Optional spatial VOT input from 4.1
INCOME_RASTER = DATA_DIR / "income_nigeria.tif"
USE_SPATIAL_VOT = True

# Output raster
OUTPUT_PIXEL_RASTER = DATA_DIR / "end_user_price.tif"

# New output band names
OUT_COST_REF_WALK = "res_cost_kg_out_walk_ref"
OUT_COST_REF_CAR = "res_cost_kg_out_car_ref"
OUT_COST_WALK = "cost_kg_walker"
OUT_COST_CAR = "cost_kg_driver"

# guards
NODATA_FLOAT = -9999.0

# Walking collection model (per trip)
WALK_TIME_IS_ONE_WAY = True
FIXED_TIME_AT_RETAILER_HOURS = 0.25
CYLINDER_WEIGHT_KG = 12.5

# Driving collection model (per trip)
CAR_VARIABLE_COST_PER_KM = 0.136


# VOT
MINIMUM_WAGE_USD_PER_MONTH = 26.0
WORK_HOURS_PER_MONTH = 30 * 8
WAGE_RANGE = (0.2, 0.5)
WAGE_FRACTION_FALLBACK = 0.3
DEFAULT_VOT_USD_PER_HOUR = WAGE_FRACTION_FALLBACK * (MINIMUM_WAGE_USD_PER_MONTH / WORK_HOURS_PER_MONTH)
VOT_TODO_NOTE = "TODO: review MINIMUM_WAGE_USD_PER_MONTH and WAGE_RANGE for local context"

# treshold for walk/car statistic
WALK_THRESHOLD_MIN = 30   
CAR_THRESHOLD_MIN = 45    





def _read_single_band(path: Path) -> tuple[np.ndarray, dict]:
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    return arr, profile


def _align_to_reference(path: Path, ref_profile: dict, resampling: Resampling) -> np.ndarray:
    with rasterio.open(path) as src:
        dst = np.full((ref_profile["height"], ref_profile["width"]), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def _compute_spatial_vot(income_array: np.ndarray) -> tuple[np.ndarray, str]:
    valid = np.isfinite(income_array)
    vot = np.full(income_array.shape, DEFAULT_VOT_USD_PER_HOUR, dtype=np.float32)
    if not np.any(valid):
        return vot, "fallback_fixed_no_valid_income"

    min_value = float(np.nanmin(income_array[valid]))
    max_value = float(np.nanmax(income_array[valid]))
    if not np.isfinite(min_value) or not np.isfinite(max_value) or max_value <= min_value:
        return vot, "fallback_fixed_flat_income"

    wage_min, wage_max = WAGE_RANGE
    norm = (income_array - min_value) / (max_value - min_value)
    norm = np.clip(norm, 0.0, 1.0)
    wage_factor = wage_min + norm * float(wage_max - wage_min)

    vot_valid = wage_factor * (MINIMUM_WAGE_USD_PER_MONTH / WORK_HOURS_PER_MONTH)
    vot[valid] = vot_valid[valid].astype(np.float32)
    return vot, "spatial_income"


def _map_effective_cost(ids_int: np.ndarray, map_resell: dict[int, float], map_filling: dict[int, float]) -> np.ndarray:
    out = np.full(ids_int.shape, np.nan, dtype=np.float64)
    valid = ids_int > 0
    if not np.any(valid):
        return out

    uniq, inv = np.unique(ids_int[valid], return_inverse=True)
    mapped = np.fromiter((map_resell.get(int(rid), np.nan) for rid in uniq), dtype=np.float64, count=uniq.size)
    miss = ~np.isfinite(mapped)
    if np.any(miss):
        miss_ids = uniq[miss]
        mapped[miss] = np.fromiter((map_filling.get(int(rid), np.nan) for rid in miss_ids), dtype=np.float64, count=miss_ids.size)

    out_valid = mapped[inv]
    out[valid] = out_valid
    return out


def _write_multiband(path: Path, base_profile: dict, bands: list[np.ndarray], names: list[str]) -> None:
    if len(bands) != len(names):
        raise ValueError("Bands and names must have same length.")
    profile = base_profile.copy()
    profile.update(dtype="float32", count=len(bands), nodata=NODATA_FLOAT, compress="lzw")

    with rasterio.open(path, "w", **profile) as dst:
        for i, (band, name) in enumerate(zip(bands, names), start=1):
            out = np.where(np.isfinite(band), band, NODATA_FLOAT).astype(np.float32)
            dst.write(out, i)
            dst.set_band_description(i, name)


# ------------------------------------------------------------------------------
# Main execution
# ------------------------------------------------------------------------------
print("[1/8] Reading source multilayer pixel raster...")
if not SOURCE_PIXEL_RASTER.exists():
    raise FileNotFoundError(f"Missing input raster from 4.2: {SOURCE_PIXEL_RASTER}")

with rasterio.open(SOURCE_PIXEL_RASTER) as src:
    # Expect 8 bands: 1:car_share, 2:walk_share, 3:walk_id, 4:walk_time, 5:walk_dist,
    #                 6:car_id, 7:car_time, 8:car_dist
    if src.count < 8:
        raise RuntimeError(f"Expected at least 8 bands in {SOURCE_PIXEL_RASTER}, found {src.count}")
    base_stack = src.read(indexes=[1, 2, 3, 4, 5, 6, 7, 8]).astype(np.float32, copy=False)
    base_profile = src.profile.copy()
    src_nodata = src.nodata

if src_nodata is not None:
    base_stack = np.where(base_stack == src_nodata, np.nan, base_stack).astype(np.float32)

car_share = base_stack[0]
walk_share = base_stack[1]
walk_id = base_stack[2]
walk_time_min = base_stack[3]
walk_dist_km = base_stack[4]        # <-- NEW: band 5
car_id = base_stack[5]              # <-- index shift: was 4, now 5
car_time_min = base_stack[6]        # was 5, now 6
car_dist_km = base_stack[7]         # <-- NEW: band 8

height, width = car_share.shape
print(f"Source grid: {width} x {height}")

print("[2/8] Checking external inputs...")
if not COST_SOURCE_GPKG.exists():
    raise FileNotFoundError(f"Missing cost source GPKG: {COST_SOURCE_GPKG}")
if not LPG_USE_SHARE_RASTER.exists():
    raise FileNotFoundError(f"Missing 4.3 output raster: {LPG_USE_SHARE_RASTER}")
if not POPULATION_RASTER.exists():
    raise FileNotFoundError(f"Missing population raster: {POPULATION_RASTER}")

print("[3/8] Loading cost layers and creating fallback cost maps...")
resell = gpd.read_file(COST_SOURCE_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(COST_SOURCE_GPKG, layer=FILLING_LAYER)
if resell.empty or filling.empty:
    raise RuntimeError("Resell or filling layer is empty in chain_with_cost.gpkg")
for c in [RESELLER_ID_COL, RESELLER_COST_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing '{c}' in layer '{RESELL_LAYER}'")
for c in [RESELLER_ID_COL, FILLING_COST_COL]:
    if c not in filling.columns:
        raise KeyError(f"Missing '{c}' in layer '{FILLING_LAYER}'")

resell_id = pd.to_numeric(resell[RESELLER_ID_COL], errors="coerce")
resell_cost = pd.to_numeric(resell[RESELLER_COST_COL], errors="coerce")
fill_id = pd.to_numeric(filling[RESELLER_ID_COL], errors="coerce")
fill_cost = pd.to_numeric(filling[FILLING_COST_COL], errors="coerce")

mask_resell = resell_id.notna()
mask_fill = fill_id.notna()
map_resell_cost = dict(zip(resell_id[mask_resell].astype(np.int64), resell_cost[mask_resell].astype(float)))
map_fill_cost = dict(zip(fill_id[mask_fill].astype(np.int64), fill_cost[mask_fill].astype(float)))

print("[3b/8] Building filling reference maps for pixel assignment...")
resell_gdf = gpd.read_file(COST_SOURCE_GPKG, layer=RESELL_LAYER)
if RESELLER_ID_COL not in resell_gdf.columns:
    raise KeyError(f"Missing '{RESELLER_ID_COL}' in resell layer")
if "filling_reference" not in resell_gdf.columns:
    raise KeyError("Missing 'filling_reference' column in resell layer (run 4.4 first)")

resell_id_series = pd.to_numeric(resell_gdf[RESELLER_ID_COL], errors="coerce")
fill_ref_series = pd.to_numeric(resell_gdf["filling_reference"], errors="coerce")
mask_resell = resell_id_series.notna() & fill_ref_series.notna()
map_reseller_to_filling = dict(zip(
    resell_id_series[mask_resell].astype(np.int64),
    fill_ref_series[mask_resell].astype(np.int64)
))

filling_gdf = gpd.read_file(COST_SOURCE_GPKG, layer=FILLING_LAYER)
fill_id_series = pd.to_numeric(filling_gdf[RESELLER_ID_COL], errors="coerce")
fill_cost_out_series = pd.to_numeric(filling_gdf[FILLING_COST_COL], errors="coerce")
mask_fill = fill_id_series.notna() & fill_cost_out_series.notna()
map_filling_cost = dict(zip(
    fill_id_series[mask_fill].astype(np.int64),
    fill_cost_out_series[mask_fill].astype(float)
))

def resolve_filling_id(rid: int) -> int:
    if rid in map_reseller_to_filling:
        return map_reseller_to_filling[rid]
    return rid

def get_filling_cost(fid: int) -> float:
    return map_filling_cost.get(fid, np.nan)

print(f"Reseller->filling mapping built: {len(map_reseller_to_filling):,} entries")
print(f"Filling cost map built: {len(map_filling_cost):,} entries")

print("[4/8] Loading population/LPG/VOT rasters...")
lpg_use_share, _ = _read_single_band(LPG_USE_SHARE_RASTER)
population, pop_profile = _read_single_band(POPULATION_RASTER)
if lpg_use_share.shape != (height, width) or population.shape != (height, width):
    raise RuntimeError("Shape mismatch among source pixel raster, LPG use raster, and population raster.")

if USE_SPATIAL_VOT and INCOME_RASTER.exists():
    income_aligned = _align_to_reference(INCOME_RASTER, pop_profile, Resampling.bilinear)
    vot_raster, vot_mode = _compute_spatial_vot(income_aligned)
elif USE_SPATIAL_VOT and (not INCOME_RASTER.exists()):
    vot_raster = np.full((height, width), DEFAULT_VOT_USD_PER_HOUR, dtype=np.float32)
    vot_mode = "fallback_fixed_income_missing"
else:
    vot_raster = np.full((height, width), DEFAULT_VOT_USD_PER_HOUR, dtype=np.float32)
    vot_mode = "fixed_user_parameter"

print("[5/8] Computing reference costs and channel demands...")
n = height * width

walk_id_flat = walk_id.reshape(-1).astype(np.float64)
car_id_flat = car_id.reshape(-1).astype(np.float64)
walk_time_flat = walk_time_min.reshape(-1).astype(np.float64)
car_time_flat = car_time_min.reshape(-1).astype(np.float64)
walk_dist_flat = walk_dist_km.reshape(-1).astype(np.float64)    # <-- NEW
car_dist_flat = car_dist_km.reshape(-1).astype(np.float64)      # <-- NEW
share_walk_flat = walk_share.reshape(-1).astype(np.float64)
share_car_flat = car_share.reshape(-1).astype(np.float64)
pop_flat = population.reshape(-1).astype(np.float64)
lpg_flat = lpg_use_share.reshape(-1).astype(np.float64)
vot_flat = vot_raster.reshape(-1).astype(np.float64)

with np.errstate(invalid='ignore'):
    walk_id_int = np.where(np.isfinite(walk_id_flat) & (walk_id_flat > 0), walk_id_flat.astype(np.int64), -1)
    car_id_int = np.where(np.isfinite(car_id_flat) & (car_id_flat > 0), car_id_flat.astype(np.int64), -1)

ref_cost_walk_arr = _map_effective_cost(walk_id_int, map_resell_cost, map_fill_cost)
ref_cost_car_arr = _map_effective_cost(car_id_int, map_resell_cost, map_fill_cost)

# Arrays for filling id and filling cost (unchanged)
filling_id_walk_arr = np.full(n, -1, dtype=np.int32)
filling_id_car_arr = np.full(n, -1, dtype=np.int32)
filling_cost_walk_arr = np.full(n, np.nan, dtype=np.float64)
filling_cost_car_arr = np.full(n, np.nan, dtype=np.float64)

valid_walk_id = (walk_id_int > 0)
if np.any(valid_walk_id):
    uniq_ids, inv = np.unique(walk_id_int[valid_walk_id], return_inverse=True)
    fid_mapped = np.array([resolve_filling_id(int(rid)) for rid in uniq_ids], dtype=np.int32)
    fcost_mapped = np.array([get_filling_cost(int(fid)) for fid in fid_mapped], dtype=np.float64)
    filling_id_walk_arr[valid_walk_id] = fid_mapped[inv]
    filling_cost_walk_arr[valid_walk_id] = fcost_mapped[inv]

valid_car_id = (car_id_int > 0)
if np.any(valid_car_id):
    uniq_ids, inv = np.unique(car_id_int[valid_car_id], return_inverse=True)
    fid_mapped = np.array([resolve_filling_id(int(rid)) for rid in uniq_ids], dtype=np.int32)
    fcost_mapped = np.array([get_filling_cost(int(fid)) for fid in fid_mapped], dtype=np.float64)
    filling_id_car_arr[valid_car_id] = fid_mapped[inv]
    filling_cost_car_arr[valid_car_id] = fcost_mapped[inv]

# Normalize walk/car shares
share_walk_arr = np.clip(share_walk_flat, 0.0, 1.0)
share_car_arr = np.clip(share_car_flat, 0.0, 1.0)
share_sum = share_walk_arr + share_car_arr
share_valid = share_sum > 0
share_walk_norm = np.zeros(n, dtype=np.float64)
share_car_norm = np.zeros(n, dtype=np.float64)
share_walk_norm[share_valid] = share_walk_arr[share_valid] / share_sum[share_valid]
share_car_norm[share_valid] = share_car_arr[share_valid] / share_sum[share_valid]

walk_frac = share_walk_norm
car_frac = share_car_norm

# ------------------------------------------------------------------------
# Walker model – per-trip collection cost
# ------------------------------------------------------------------------
# Assumption: time is one-way; round-trip = 2 * time_min + fixed retailer time.
walk_round_trip_hours = np.where(
    WALK_TIME_IS_ONE_WAY,
    (2.0 * walk_time_flat) / 60.0,
    walk_time_flat / 60.0
)
total_trip_time_hours_walk = walk_round_trip_hours + FIXED_TIME_AT_RETAILER_HOURS
trip_cost_walk_usd = total_trip_time_hours_walk * vot_flat
collection_cost_per_kg_walk = trip_cost_walk_usd / CYLINDER_WEIGHT_KG

valid_walk_mask = (
    np.isfinite(ref_cost_walk_arr)
    & (walk_id_int > 0)
    & np.isfinite(walk_round_trip_hours)
    & (walk_round_trip_hours >= 0)
    & np.isfinite(vot_flat)
)

cost_walk_final = np.full(n, np.nan, dtype=np.float64)
cost_walk_final[valid_walk_mask] = (
    ref_cost_walk_arr[valid_walk_mask] + collection_cost_per_kg_walk[valid_walk_mask]
)

# ------------------------------------------------------------------------
# Driver model – per-trip collection cost using actual distances
# ------------------------------------------------------------------------
round_trip_distance_km_car = 2.0 * car_dist_flat
round_trip_drive_hours = (car_time_flat * 2.0) / 60.0
total_trip_time_hours_car = round_trip_drive_hours + FIXED_TIME_AT_RETAILER_HOURS

vehicle_operating_cost_trip_usd = round_trip_distance_km_car * CAR_VARIABLE_COST_PER_KM
driver_time_cost_trip_usd = total_trip_time_hours_car * vot_flat
trip_cost_car_usd = vehicle_operating_cost_trip_usd + driver_time_cost_trip_usd
collection_cost_per_kg_car = trip_cost_car_usd / CYLINDER_WEIGHT_KG

valid_driver_mask = (
    np.isfinite(ref_cost_car_arr)
    & (car_id_int > 0)
    & np.isfinite(car_time_flat)
    & (car_time_flat >= 0)
    & np.isfinite(car_dist_flat)          # <-- ensure distance is valid
    & (car_dist_flat >= 0)
    & np.isfinite(total_trip_time_hours_car)
    & np.isfinite(vot_flat)
)

cost_car_final = np.full(n, np.nan, dtype=np.float64)
cost_car_final[valid_driver_mask] = (
    ref_cost_car_arr[valid_driver_mask] + collection_cost_per_kg_car[valid_driver_mask]
)

# new band with mean cost per kg weighted by walk/car shares (NaN if no valid cost or zero share)
mean_user_cost_flat = np.full(n, np.nan, dtype=np.float64)
valid_mean = (share_walk_norm + share_car_norm) > 0
mean_user_cost_flat[valid_mean] = (
    cost_car_final[valid_mean] * share_car_norm[valid_mean] +
    cost_walk_final[valid_mean] * share_walk_norm[valid_mean]
)
mean_user_cost = mean_user_cost_flat.reshape(height, width)



# ------------------------------------------------------------------------
print("[6/8] Writing output multilayer raster...")

best_id_walk_band = np.where(np.isfinite(walk_id) & (walk_id >= 0), walk_id, np.nan)
best_id_car_band = np.where(np.isfinite(car_id) & (car_id >= 0), car_id, np.nan)

out_ref_walk = ref_cost_walk_arr.reshape(height, width)
out_ref_car = ref_cost_car_arr.reshape(height, width)
out_cost_walk = cost_walk_final.reshape(height, width)
out_cost_car = cost_car_final.reshape(height, width)

out_bands = [
    car_share.astype(np.float32),
    walk_share.astype(np.float32),
    best_id_walk_band.astype(np.float32),
    walk_time_min.astype(np.float32),
    walk_dist_km.astype(np.float32),                     # <-- NUOVO
    best_id_car_band.astype(np.float32),
    car_time_min.astype(np.float32),
    car_dist_km.astype(np.float32),                      # <-- NUOVO
    out_ref_walk.astype(np.float32),
    out_ref_car.astype(np.float32),
    out_cost_walk.astype(np.float32),
    out_cost_car.astype(np.float32),
    filling_id_walk_arr.reshape(height, width).astype(np.float32),
    filling_id_car_arr.reshape(height, width).astype(np.float32),
    filling_cost_walk_arr.reshape(height, width).astype(np.float32),
    filling_cost_car_arr.reshape(height, width).astype(np.float32),
    mean_user_cost.astype(np.float32),
]

out_names = [
    "car_share",
    "walk_share",
    "best_reseller_id_walk",
    "best_time_walk_min",
    "best_distance_walk_km",           # <-- NUOVO
    "best_reseller_id_car",
    "best_time_car_min",
    "best_distance_car_km",            # <-- NUOVO
    OUT_COST_REF_WALK,
    OUT_COST_REF_CAR,
    OUT_COST_WALK,
    OUT_COST_CAR,
    "filling_id_walk",
    "filling_id_car",
    "cost_fil_kg_out_walk_ref",
    "cost_fil_kg_out_car_ref",
    "mean_user_cost"
]

_write_multiband(OUTPUT_PIXEL_RASTER, base_profile, out_bands, out_names)

# ------------------------------------------------------------------------
print("[7/8] Diagnostics...")
n_total = n
n_valid_walk = int(valid_walk_mask.sum())
n_nan_walk = int(np.isnan(cost_walk_final).sum())
n_valid_driver = int(valid_driver_mask.sum())
n_nan_driver = int(np.isnan(cost_car_final).sum())

print("=== WALKER COST MODEL SUMMARY ===")
print(f"Rows total                                 : {n_total:,}")
print(f"Rows valid walker cost                     : {n_valid_walk:,}/{n_total:,}")
print(f"Rows NaN (no cost)                         : {n_nan_walk:,}/{n_total:,}")
print("Collection cost method                      : per-trip / cylinder (12.5 kg)")

print("\n=== DRIVER COST MODEL SUMMARY ===")
print(f"Rows total                                 : {n_total:,}")
print(f"Rows valid driver cost                     : {n_valid_driver:,}/{n_total:,}")
print(f"Rows NaN (no cost)                         : {n_nan_driver:,}/{n_total:,}")
print("Collection cost method                      : per-trip / cylinder (12.5 kg)")
print(f"Car variable cost (USD/km)                  : {CAR_VARIABLE_COST_PER_KM}")

print(f"\nVOT mode                                   : {vot_mode}")
if np.any(np.isfinite(vot_flat)):
    v = vot_flat[np.isfinite(vot_flat)]
    print(f"VOT USD/h min/median/mean/p95/max         : {np.nanmin(v):.6f} / {np.nanmedian(v):.6f} / {np.nanmean(v):.6f} / {np.nanpercentile(v, 95):.6f} / {np.nanmax(v):.6f}")
print(VOT_TODO_NOTE)

n_walk_id_valid = int(np.sum(walk_id_int > 0))
n_car_id_valid = int(np.sum(car_id_int > 0))
n_walk_id_missing = int(np.sum(walk_id_int == -1))
n_car_id_missing = int(np.sum(car_id_int == -1))

print("=== ID ASSIGNMENT DIAGNOSTICS ===")
print(f"Total pixels                     : {n_total:,}")
print(f"Walk ID valid (>0)               : {n_walk_id_valid:,} ({100*n_walk_id_valid/n_total:.2f}%)")
print(f"Car ID valid (>0)                : {n_car_id_valid:,} ({100*n_car_id_valid/n_total:.2f}%)")
print(f"Walk ID missing (-1)             : {n_walk_id_missing:,} ({100*n_walk_id_missing/n_total:.2f}%)")
print(f"Car ID missing (-1)              : {n_car_id_missing:,} ({100*n_car_id_missing/n_total:.2f}%)")

print("=== FILLING ID ASSIGNMENT ===")
print(f"Walk pixels with valid filling id: {np.sum(filling_id_walk_arr > 0):,}")
print(f"Car pixels with valid filling id : {np.sum(filling_id_car_arr > 0):,}")
print(f"Unique filling ids (walk)        : {len(np.unique(filling_id_walk_arr[filling_id_walk_arr > 0]))}")
print(f"Unique filling ids (car)         : {len(np.unique(filling_id_car_arr[filling_id_car_arr > 0]))}")


print("\n=== MEAN USER COST (weighted) STATISTICS ===")
valid_mean_finite = np.isfinite(mean_user_cost_flat)
if np.any(valid_mean_finite):
    m = mean_user_cost_flat[valid_mean_finite]
    print(f"Mean user cost USD/kg  : min={np.nanmin(m):.6f} / median={np.nanmedian(m):.6f} / "
          f"mean={np.nanmean(m):.6f} / p95={np.nanpercentile(m, 95):.6f} / max={np.nanmax(m):.6f}")
else:
    print("No valid mean user cost pixels.")

#stats for accessibility within time threshold
print("\n=== WALKER COST WITH TIME <= {} min ===".format(WALK_THRESHOLD_MIN))
walk_within_thresh = (valid_walk_mask) & (walk_time_flat <= WALK_THRESHOLD_MIN)
if np.any(walk_within_thresh):
    cw = cost_walk_final[walk_within_thresh]
    print(f"Walker cost USD/kg (time <= {WALK_THRESHOLD_MIN} min): "
          f"min={np.nanmin(cw):.6f} / median={np.nanmedian(cw):.6f} / "
          f"mean={np.nanmean(cw):.6f} / max={np.nanmax(cw):.6f}")
    print(f"Pixels in this group: {np.sum(walk_within_thresh):,}")
else:
    print("No walker pixels within time threshold.")

print("\n=== DRIVER COST WITH TIME <= {} min ===".format(CAR_THRESHOLD_MIN))
car_within_thresh = (valid_driver_mask) & (car_time_flat <= CAR_THRESHOLD_MIN)
if np.any(car_within_thresh):
    cc = cost_car_final[car_within_thresh]
    print(f"Driver cost USD/kg (time <= {CAR_THRESHOLD_MIN} min): "
          f"min={np.nanmin(cc):.6f} / median={np.nanmedian(cc):.6f} / "
          f"mean={np.nanmean(cc):.6f} / max={np.nanmax(cc):.6f}")
    print(f"Pixels in this group: {np.sum(car_within_thresh):,}")
else:
    print("No driver pixels within time threshold.")




print("[8/8] Done.")
print(f"Output file: {OUTPUT_PIXEL_RASTER}")
print(f"Bands written: {', '.join(out_names)}")
print("Walker formula (per trip):")
print("  cost_kg_walker = reference_point_cost + (round_trip_time_h * VOT) / 12.5")
print("Driver formula (per trip):")
print("  cost_kg_driver = reference_point_cost + (vehicle_op_cost + driver_time_cost) / 12.5")
print(f"Primary reseller source column: {RESELLER_COST_COL}")
print(f"Fallback filling source column: {FILLING_COST_COL}")

[1/8] Reading source multilayer pixel raster...
Source grid: 1337 x 1085
[2/8] Checking external inputs...
[3/8] Loading cost layers and creating fallback cost maps...
[3b/8] Building filling reference maps for pixel assignment...
Reseller->filling mapping built: 2,413 entries
Filling cost map built: 375 entries
[4/8] Loading population/LPG/VOT rasters...
[5/8] Computing reference costs and channel demands...
[6/8] Writing output multilayer raster...
[7/8] Diagnostics...
=== WALKER COST MODEL SUMMARY ===
Rows total                                 : 1,450,645
Rows valid walker cost                     : 559,129/1,450,645
Rows NaN (no cost)                         : 891,516/1,450,645
Collection cost method                      : per-trip / cylinder (12.5 kg)

=== DRIVER COST MODEL SUMMARY ===
Rows total                                 : 1,450,645
Rows valid driver cost                     : 558,600/1,450,645
Rows NaN (no cost)                         : 892,045/1,450,645
Collection cost m

In [2]:
"""
Price variation analysis across pixels assigned to the same retailer (ID).
"""

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

DATA_DIR = Path("dataset_big")
OUTPUT_RASTER = DATA_DIR / "end_user_price.tif"
NODATA_FLOAT = -9999.0

print("Loading output raster...")
with rasterio.open(OUTPUT_RASTER) as src:
    arr = src.read()
    desc = list(src.descriptions)
    nodata = src.nodata

if nodata is None:
    nodata = NODATA_FLOAT

arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
name_to_idx = {d: i for i, d in enumerate(desc) if d is not None}

cost_walk = arr[name_to_idx["cost_kg_walker"]].flatten()
cost_car = arr[name_to_idx["cost_kg_driver"]].flatten()
ref_walk = arr[name_to_idx["res_cost_kg_out_walk_ref"]].flatten()
ref_car = arr[name_to_idx["res_cost_kg_out_car_ref"]].flatten()
walk_id = arr[name_to_idx["best_reseller_id_walk"]].flatten()
car_id = arr[name_to_idx["best_reseller_id_car"]].flatten()

valid_walk = (walk_id > 0) & np.isfinite(cost_walk)
valid_car = (car_id > 0) & np.isfinite(cost_car)

df_walk = pd.DataFrame({
    'retailer_id': walk_id[valid_walk].astype(int),
    'cost_final': cost_walk[valid_walk],
    'cost_ref': ref_walk[valid_walk],
    'collection_component': cost_walk[valid_walk] - ref_walk[valid_walk]
})

df_car = pd.DataFrame({
    'retailer_id': car_id[valid_car].astype(int),
    'cost_final': cost_car[valid_car],
    'cost_ref': ref_car[valid_car],
    'collection_component': cost_car[valid_car] - ref_car[valid_car]
})

def retailer_stats(df, mode_name):
    stats = df.groupby('retailer_id').agg(
        num_pixels=('cost_final', 'count'),
        cost_min=('cost_final', 'min'),
        cost_max=('cost_final', 'max'),
        ref_cost=('cost_ref', 'first'),
        component_max=('collection_component', 'max'),
        component_min=('collection_component', 'min')
    ).reset_index()
    stats['cost_range'] = stats['cost_max'] - stats['cost_min']
    stats['component_range'] = stats['component_max'] - stats['component_min']
    stats_sorted = stats.sort_values('cost_range', ascending=False)

    print(f"\n=== {mode_name.upper()} MODE STATISTICS ===")
    print(f"Unique retailers: {len(stats)}")
    print("Final cost range (max-min):")
    print(f"  Min:    {stats['cost_range'].min():.6f}")
    print(f"  Median: {stats['cost_range'].median():.6f}")
    print(f"  Max:    {stats['cost_range'].max():.6f}")
    print("Top 5 retailers with largest cost spread:")
    for _, row in stats_sorted.head(5).iterrows():
        print(f"  ID {int(row['retailer_id'])}: pixels={int(row['num_pixels'])}, "
              f"ref cost={row['ref_cost']:.4f}, range={row['cost_range']:.6f}")
    return stats

stats_walk = retailer_stats(df_walk, "WALK")
stats_car = retailer_stats(df_car, "CAR")

Loading output raster...

=== WALK MODE STATISTICS ===
Unique retailers: 2107
Final cost range (max-min):
  Min:    0.000000
  Median: 0.006450
  Max:    0.241787
Top 5 retailers with largest cost spread:
  ID 2275: pixels=12683, ref cost=1.2608, range=0.241787
  ID 113: pixels=422, ref cost=1.1650, range=0.200675
  ID 1901: pixels=1616, ref cost=1.2589, range=0.196086
  ID 266: pixels=327, ref cost=1.2400, range=0.181212
  ID 105: pixels=476, ref cost=1.1674, range=0.175852

=== CAR MODE STATISTICS ===
Unique retailers: 2107
Final cost range (max-min):
  Min:    0.000000
  Median: 0.096744
  Max:    5.564259
Top 5 retailers with largest cost spread:
  ID 2267: pixels=14229, ref cost=1.2646, range=5.564259
  ID 1568: pixels=4758, ref cost=1.2671, range=5.143638
  ID 2272: pixels=852, ref cost=1.2598, range=4.469335
  ID 2263: pixels=1525, ref cost=1.2598, range=4.434209
  ID 1758: pixels=7069, ref cost=1.3185, range=4.211718


In [3]:
"""
Verify that NaN pixels in final cost correspond to pixels without population.
"""

from pathlib import Path
import numpy as np
import rasterio

DATA_DIR = Path("dataset_big")
OUTPUT_RASTER = DATA_DIR / "end_user_price.tif"
POP_RASTER = DATA_DIR / "Population.tif"
NODATA_FLOAT = -9999.0

with rasterio.open(OUTPUT_RASTER) as src:
    desc = list(src.descriptions)
    idx_walk = desc.index("cost_kg_walker")
    idx_car = desc.index("cost_kg_driver")
    cost_walk = src.read(idx_walk + 1).astype(np.float32)
    cost_car = src.read(idx_car + 1).astype(np.float32)
    nodata = src.nodata

with rasterio.open(POP_RASTER) as src_pop:
    pop = src_pop.read(1).astype(np.float32)
    pop_nodata = src_pop.nodata

if nodata is not None:
    cost_walk = np.where(cost_walk == nodata, np.nan, cost_walk)
    cost_car = np.where(cost_car == nodata, np.nan, cost_car)
if pop_nodata is not None:
    pop = np.where(pop == pop_nodata, np.nan, pop)

pop_zero_or_nan = ~np.isfinite(pop) | (pop == 0)
pop_positive = np.isfinite(pop) & (pop > 0)

walk_nan = ~np.isfinite(cost_walk)
car_nan = ~np.isfinite(cost_car)

walk_nan_in_pop_positive = (walk_nan & pop_positive).sum()
car_nan_in_pop_positive = (car_nan & pop_positive).sum()

print("=== NaN vs POPULATION CHECK ===")
print(f"Walk pixels NaN where pop > 0: {walk_nan_in_pop_positive:,}")
print(f"Car pixels NaN where pop > 0 : {car_nan_in_pop_positive:,}")
if walk_nan_in_pop_positive == 0 and car_nan_in_pop_positive == 0:
    print("✅ All NaN values correspond to pixels without population.")
else:
    print("⚠️ There are populated pixels without a cost (possible missing retailers).")

=== NaN vs POPULATION CHECK ===
Walk pixels NaN where pop > 0: 4,723
Car pixels NaN where pop > 0 : 5,252
⚠️ There are populated pixels without a cost (possible missing retailers).


In [4]:
"""
Cost decomposition analysis: filling cost, reseller markup, and collection cost.
Plus accessibility analysis for mean, walk, and car costs.
"""

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

DATA_DIR = Path("dataset_big")
OUTPUT_RASTER = DATA_DIR / "end_user_price.tif"
NODATA_FLOAT = -9999.0

# Soglie di accessibilità
WALK_THRESHOLD_MIN = 30   # minuti a piedi
CAR_THRESHOLD_MIN = 45    # minuti in auto

print("Loading output raster for cost decomposition...")
with rasterio.open(OUTPUT_RASTER) as src:
    arr = src.read()
    desc = list(src.descriptions)
    nodata = src.nodata

if nodata is None:
    nodata = NODATA_FLOAT

arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
name_to_idx = {d: i for i, d in enumerate(desc) if d is not None}

# Required bands
required = [
    "cost_kg_walker",
    "res_cost_kg_out_walk_ref",
    "cost_fil_kg_out_walk_ref",
    "cost_kg_driver",
    "res_cost_kg_out_car_ref",
    "cost_fil_kg_out_car_ref",
    "mean_user_cost",          # NUOVA banda
    "best_time_walk_min",      # per accessibilità walk
    "best_time_car_min",       # per accessibilità car
    "walk_share",              # per pesi
    "car_share"                # per pesi
]

for band in required:
    if band not in name_to_idx:
        raise KeyError(f"Band '{band}' not found in raster. Run 4.6 with the latest version.")

# Extract flat arrays
cost_walk = arr[name_to_idx["cost_kg_walker"]].flatten()
ref_walk = arr[name_to_idx["res_cost_kg_out_walk_ref"]].flatten()
fill_walk = arr[name_to_idx["cost_fil_kg_out_walk_ref"]].flatten()

cost_car = arr[name_to_idx["cost_kg_driver"]].flatten()
ref_car = arr[name_to_idx["res_cost_kg_out_car_ref"]].flatten()
fill_car = arr[name_to_idx["cost_fil_kg_out_car_ref"]].flatten()

mean_cost = arr[name_to_idx["mean_user_cost"]].flatten()
walk_time = arr[name_to_idx["best_time_walk_min"]].flatten()
car_time = arr[name_to_idx["best_time_car_min"]].flatten()
walk_share = arr[name_to_idx["walk_share"]].flatten()
car_share_arr = arr[name_to_idx["car_share"]].flatten()

# Valid pixels: finite values for all components
valid_walk = (
    np.isfinite(cost_walk) &
    np.isfinite(ref_walk) &
    np.isfinite(fill_walk)
)
valid_car = (
    np.isfinite(cost_car) &
    np.isfinite(ref_car) &
    np.isfinite(fill_car)
)

# Valid mean cost pixels (almeno un modo di trasporto ha peso >0)
valid_mean = np.isfinite(mean_cost)

# Compute decomposition
reseller_markup_walk = ref_walk - fill_walk
collection_walk = cost_walk - ref_walk

reseller_markup_car = ref_car - fill_car
collection_car = cost_car - ref_car

def print_stats(name, arr, mask):
    if not np.any(mask):
        print(f"{name}: no valid pixels")
        return
    data = arr[mask]
    print(f"{name}: min={np.nanmin(data):.6f} / median={np.nanmedian(data):.6f} / "
          f"mean={np.nanmean(data):.6f} / p95={np.nanpercentile(data, 95):.6f} / max={np.nanmax(data):.6f}")

print("\n=== COST DECOMPOSITION – WALK MODE ===")
print(f"Valid walk pixels: {np.sum(valid_walk):,}")
print_stats("  Final cost (cost_kg_walker)      ", cost_walk, valid_walk)
print_stats("  Filling wholesale (cost_fil_kg_out)", fill_walk, valid_walk)
print_stats("  Reseller markup                  ", reseller_markup_walk, valid_walk)
print_stats("  Collection cost                  ", collection_walk, valid_walk)

print("\n=== COST DECOMPOSITION – CAR MODE ===")
print(f"Valid car pixels: {np.sum(valid_car):,}")
print_stats("  Final cost (cost_kg_driver)      ", cost_car, valid_car)
print_stats("  Filling wholesale (cost_fil_kg_out)", fill_car, valid_car)
print_stats("  Reseller markup                  ", reseller_markup_car, valid_car)
print_stats("  Collection cost                  ", collection_car, valid_car)

# Summary table of average shares
def share_stats(cost, fill, markup, coll, mask):
    if not np.any(mask):
        return {}
    total = cost[mask]
    return {
        "fill_pct": 100 * np.nanmean(fill[mask] / total),
        "markup_pct": 100 * np.nanmean(markup[mask] / total),
        "coll_pct": 100 * np.nanmean(coll[mask] / total),
    }

walk_shares = share_stats(cost_walk, fill_walk, reseller_markup_walk, collection_walk, valid_walk)
car_shares = share_stats(cost_car, fill_car, reseller_markup_car, collection_car, valid_car)

print("\n=== AVERAGE COST SHARE (as % of final cost) ===")
if walk_shares:
    print(f"Walk : filling={walk_shares['fill_pct']:.1f}%  "
          f"reseller_markup={walk_shares['markup_pct']:.1f}%  "
          f"collection={walk_shares['coll_pct']:.1f}%")
if car_shares:
    print(f"Car  : filling={car_shares['fill_pct']:.1f}%  "
          f"reseller_markup={car_shares['markup_pct']:.1f}%  "
          f"collection={car_shares['coll_pct']:.1f}%")


# MEAN USER COST

print("\n" + "="*60)
print("=== MEAN USER COST (weighted average) STATISTICS ===")
print("="*60)

if np.any(valid_mean):
    print_stats("  Mean user cost (USD/kg)          ", mean_cost, valid_mean)
    
    # Cost shares for mean user cost (scomposizione pesata)
    # Calcoliamo i contributi relativi dei tre componenti
    fill_contrib = np.full_like(mean_cost, np.nan)
    markup_contrib = np.full_like(mean_cost, np.nan)
    coll_contrib = np.full_like(mean_cost, np.nan)
    
    # Per ogni pixel, il costo medio pesato è la somma dei contributi pesati
    valid_for_shares = valid_mean & valid_walk & valid_car & np.isfinite(walk_share) & np.isfinite(car_share_arr)
    
    if np.any(valid_for_shares):
        # Normalizza le share (potrebbero non sommare a 1 a causa di NaN)
        share_sum = walk_share[valid_for_shares] + car_share_arr[valid_for_shares]
        walk_weight = walk_share[valid_for_shares] / share_sum
        car_weight = car_share_arr[valid_for_shares] / share_sum
        
        fill_contrib[valid_for_shares] = (
            fill_walk[valid_for_shares] * walk_weight +
            fill_car[valid_for_shares] * car_weight
        )
        markup_contrib[valid_for_shares] = (
            reseller_markup_walk[valid_for_shares] * walk_weight +
            reseller_markup_car[valid_for_shares] * car_weight
        )
        coll_contrib[valid_for_shares] = (
            collection_walk[valid_for_shares] * walk_weight +
            collection_car[valid_for_shares] * car_weight
        )
        
        valid_shares = np.isfinite(fill_contrib) & np.isfinite(markup_contrib) & np.isfinite(coll_contrib)
        
        if np.any(valid_shares):
            fill_pct_mean = 100 * np.nanmean(fill_contrib[valid_shares] / mean_cost[valid_shares])
            markup_pct_mean = 100 * np.nanmean(markup_contrib[valid_shares] / mean_cost[valid_shares])
            coll_pct_mean = 100 * np.nanmean(coll_contrib[valid_shares] / mean_cost[valid_shares])
            
            print("\n=== AVERAGE COST SHARE FOR MEAN USER COST ===")
            print(f"Mean : filling={fill_pct_mean:.1f}%  "
                  f"reseller_markup={markup_pct_mean:.1f}%  "
                  f"collection={coll_pct_mean:.1f}%")
else:
    print("No valid mean user cost pixels.")


# ACCESSIBILITY ANALYSIS

print("\n" + "="*60)
print(f"=== ACCESSIBILITY ANALYSIS (walk time <= {WALK_THRESHOLD_MIN} min) ===")
print("="*60)

walk_within_thresh = (valid_walk) & (walk_time <= WALK_THRESHOLD_MIN)
if np.any(walk_within_thresh):
    cw_within = cost_walk[walk_within_thresh]
    cw_all = cost_walk[valid_walk]
    
    print(f"\nWALKER COST - within {WALK_THRESHOLD_MIN} minutes:")
    print(f"  Pixels in group: {np.sum(walk_within_thresh):,} / {np.sum(valid_walk):,} ({100*np.sum(walk_within_thresh)/np.sum(valid_walk):.1f}%)")
    print(f"  Min:  {np.nanmin(cw_within):.6f}")
    print(f"  Median: {np.nanmedian(cw_within):.6f}")
    print(f"  Mean: {np.nanmean(cw_within):.6f}")
    print(f"  Max:  {np.nanmax(cw_within):.6f}")
    print(f"  Comparison with all walk pixels (median): {np.nanmedian(cw_within):.6f} vs {np.nanmedian(cw_all):.6f}")
    print(f"  Difference: {np.nanmedian(cw_within) - np.nanmedian(cw_all):.6f} ({(np.nanmedian(cw_within)/np.nanmedian(cw_all)-1)*100:+.1f}%)")
else:
    print(f"No walker pixels within {WALK_THRESHOLD_MIN} minutes.")

print("\n" + "="*60)
print(f"=== ACCESSIBILITY ANALYSIS (car time <= {CAR_THRESHOLD_MIN} min) ===")
print("="*60)

car_within_thresh = (valid_car) & (car_time <= CAR_THRESHOLD_MIN)
if np.any(car_within_thresh):
    cc_within = cost_car[car_within_thresh]
    cc_all = cost_car[valid_car]
    
    print(f"\nDRIVER COST - within {CAR_THRESHOLD_MIN} minutes:")
    print(f"  Pixels in group: {np.sum(car_within_thresh):,} / {np.sum(valid_car):,} ({100*np.sum(car_within_thresh)/np.sum(valid_car):.1f}%)")
    print(f"  Min:  {np.nanmin(cc_within):.6f}")
    print(f"  Median: {np.nanmedian(cc_within):.6f}")
    print(f"  Mean: {np.nanmean(cc_within):.6f}")
    print(f"  Max:  {np.nanmax(cc_within):.6f}")
    print(f"  Comparison with all car pixels (median): {np.nanmedian(cc_within):.6f} vs {np.nanmedian(cc_all):.6f}")
    print(f"  Difference: {np.nanmedian(cc_within) - np.nanmedian(cc_all):.6f} ({(np.nanmedian(cc_within)/np.nanmedian(cc_all)-1)*100:+.1f}%)")
else:
    print(f"No driver pixels within {CAR_THRESHOLD_MIN} minutes.")


# MEAN USER COST WITH ACCESSIBILITY (both types of access)

print("\n" + "="*60)
print(f"=== MEAN USER COST WITH BOTH MODES ACCESSIBLE ===")
print(f"    (walk time <= {WALK_THRESHOLD_MIN} min AND car time <= {CAR_THRESHOLD_MIN} min)")
print("="*60)

both_accessible = (valid_mean) & (walk_time <= WALK_THRESHOLD_MIN) & (car_time <= CAR_THRESHOLD_MIN)
if np.any(both_accessible):
    mean_within = mean_cost[both_accessible]
    mean_all = mean_cost[valid_mean]
    
    print(f"\nPixels with both modes accessible: {np.sum(both_accessible):,} / {np.sum(valid_mean):,} ({100*np.sum(both_accessible)/np.sum(valid_mean):.1f}%)")
    print(f"  Min:  {np.nanmin(mean_within):.6f}")
    print(f"  Median: {np.nanmedian(mean_within):.6f}")
    print(f"  Mean: {np.nanmean(mean_within):.6f}")
    print(f"  Max:  {np.nanmax(mean_within):.6f}")
    print(f"  Comparison with all mean user cost pixels (median): {np.nanmedian(mean_within):.6f} vs {np.nanmedian(mean_all):.6f}")
    print(f"  Difference: {np.nanmedian(mean_within) - np.nanmedian(mean_all):.6f} ({(np.nanmedian(mean_within)/np.nanmedian(mean_all)-1)*100:+.1f}%)")
else:
    print("No pixels with both modes accessible within thresholds.")

print("\n" + "="*60)
print("Cost decomposition and accessibility analysis completed.")
print("="*60)

Loading output raster for cost decomposition...

=== COST DECOMPOSITION – WALK MODE ===
Valid walk pixels: 559,129
  Final cost (cost_kg_walker)      : min=1.045107 / median=1.292678 / mean=1.284239 / p95=1.370127 / max=1.503305
  Filling wholesale (cost_fil_kg_out): min=1.043937 / median=1.237253 / mean=1.225129 / p95=1.281264 / max=1.294292
  Reseller markup                  : min=0.000000 / median=0.025300 / mean=0.027195 / p95=0.056282 / max=0.075382
  Collection cost                  : min=0.000475 / median=0.025796 / mean=0.031915 / p95=0.079473 / max=0.242470

=== COST DECOMPOSITION – CAR MODE ===
Valid car pixels: 558,600
  Final cost (cost_kg_driver)      : min=1.045107 / median=2.001981 / mean=2.195733 / p95=3.769201 / max=6.829925
  Filling wholesale (cost_fil_kg_out): min=1.043937 / median=1.237253 / mean=1.225493 / p95=1.281264 / max=1.294292
  Reseller markup                  : min=0.000000 / median=0.024666 / mean=0.026911 / p95=0.056282 / max=0.075382
  Collection cost 